<a href="https://colab.research.google.com/github/DiyaRana7/Flyrank_ML/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [18]:
import os
import pandas as pd
import numpy as np

os.chdir("/content/Flyrank_ML")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Working directory:", os.getcwd())
print("Dataset shape:", df.shape)
print("Dataset loaded successfully.")

Working directory: /content/Flyrank_ML
Dataset shape: (30000, 44)
Dataset loaded successfully.


In [26]:
# Signal check 1: staleness vs page visibility

df["stale_bucket"] = np.where(
    df["days_since_last_update"] >= 180,
    "stale_180_plus",
    "recent_under_180"
)

staleness_check = (
    df.groupby("stale_bucket")
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          mean_impressions=("impressions_90d", "mean")
      )
      .reset_index()
)

display(staleness_check)

print("VERDICT: MIXED")
print(
    "I will treat staleness as a directional prioritization signal, "
    "not proof that an old page needs a refresh."
)

,stale_bucket,n,median_impressions,mean_impressions
0,recent_under_180,29826,742.0,5223.864514
1,stale_180_plus,174,15.5,1172.448276


VERDICT: MIXED
I will treat staleness as a directional prioritization signal, not proof that an old page needs a refresh.


In [27]:
# Signal check 2: average position vs CTR

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, 100],
    labels=["top_3", "positions_4_10", "positions_11_20", "positions_21_plus"],
    include_lowest=True
)

position_check = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          mean_ctr=("ctr", "mean")
      )
      .reset_index()
)

display(position_check)

print("VERDICT: CONFIRMED")
print(
    "The observed CTR differs across position buckets, "
    "so position is a reasonable directional signal for prioritization."
)

,position_bucket,n,mean_ctr
0,top_3,2346,1.472869
1,positions_4_10,11842,0.651045
2,positions_11_20,7273,0.323443
3,positions_21_plus,8524,0.211705


VERDICT: CONFIRMED
The observed CTR differs across position buckets, so position is a reasonable directional signal for prioritization.


### Signal-check conclusion

The two checks provide directional support for using staleness, visibility, and position as baseline signals.

The signal checks do not prove that a page needs refreshing. They only show that these observable signals have measurable differences across groups in the starter data.

I therefore use them as prioritization signals for a human-review queue rather than as automatic decisions.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

I want to prioritize pages for possible content-refresh review using observable signals available at the decision moment.

A page receives points when:

- `days_since_last_update >= 180` → +2 points
- `days_since_last_update >= 365` → +1 additional point
- `impressions_90d >= 500` → +2 points
- `impressions_90d >= 5000` → +1 additional point
- `avg_position > 10` → +1 point

Pages with a score of 4 or more receive the action label `REVIEW_REFRESH`.
Pages with a lower score receive `MONITOR`.

### Reason codes

- `STALE_VISIBLE` — the page is at least 180 days old and has at least 500 impressions.
- `STALE` — the page is at least 180 days old but does not meet the visibility threshold.
- `VISIBLE` — the page has at least 500 impressions but is not stale.
- `OTHER` — the page does not meet the main priority conditions.

This is a simple baseline for decision support. A high score means that a page is worth human review; it does not prove that the page needs a refresh or that refreshing it will improve performance.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
# Create a working copy
baseline = df.copy()

# Start score at zero
baseline["score"] = 0

# Staleness signals
baseline["score"] += (
    baseline["days_since_last_update"] >= 180
).astype(int) * 2

baseline["score"] += (
    baseline["days_since_last_update"] >= 365
).astype(int) * 1

# Visibility signals
baseline["score"] += (
    baseline["impressions_90d"] >= 500
).astype(int) * 2

baseline["score"] += (
    baseline["impressions_90d"] >= 5000
).astype(int) * 1

# Position signal
baseline["score"] += (
    baseline["avg_position"] > 10
).astype(int) * 1

print("Score created successfully.")

display(
    baseline[
        [
            "content_id",
            "days_since_last_update",
            "impressions_90d",
            "avg_position",
            "score"
        ]
    ].head(10)
)

Score created successfully.


,content_id,days_since_last_update,impressions_90d,avg_position,score
0,content_304f48230142,20,3803,10.6,3
1,content_a1fb4e703a9e,25,15320,20.3,4
2,content_9aa793d4d895,20,12581,36.5,4
3,content_331d6c4de07b,22,11751,6.2,3
4,content_d99b7a2d90ca,14,19140,44.0,4
5,content_d4084a4bc775,20,3970,8.5,2
6,content_9a34b442b552,20,20,7.0,0
7,content_a63219c6e95a,22,1724,21.2,3
8,content_5e6c160719bc,20,32574,46.0,4
9,content_c27558df2b0c,104,1240,4.9,2


In [20]:
# Create reason codes
baseline["reason_code"] = np.select(
    [
        (baseline["days_since_last_update"] >= 180) &
        (baseline["impressions_90d"] >= 500),

        baseline["days_since_last_update"] >= 180,

        baseline["impressions_90d"] >= 500
    ],
    [
        "STALE_VISIBLE",
        "STALE",
        "VISIBLE"
    ],
    default="OTHER"
)

# Create action label
baseline["action"] = np.where(
    baseline["score"] >= 4,
    "REVIEW_REFRESH",
    "MONITOR"
)

# Create confidence note
baseline["confidence_note"] = np.select(
    [
        baseline["score"] >= 6,
        baseline["score"] >= 4
    ],
    [
        "Multiple strong priority signals",
        "Meets baseline review threshold"
    ],
    default="Lower priority based on baseline"
)

print("Reason codes and actions created.")

display(
    baseline[
        [
            "content_id",
            "score",
            "reason_code",
            "action",
            "confidence_note"
        ]
    ].head(10)
)

Reason codes and actions created.


,content_id,score,reason_code,action,confidence_note
0,content_304f48230142,3,VISIBLE,MONITOR,Lower priority based on baseline
1,content_a1fb4e703a9e,4,VISIBLE,REVIEW_REFRESH,Meets baseline review threshold
2,content_9aa793d4d895,4,VISIBLE,REVIEW_REFRESH,Meets baseline review threshold
3,content_331d6c4de07b,3,VISIBLE,MONITOR,Lower priority based on baseline
4,content_d99b7a2d90ca,4,VISIBLE,REVIEW_REFRESH,Meets baseline review threshold
5,content_d4084a4bc775,2,VISIBLE,MONITOR,Lower priority based on baseline
6,content_9a34b442b552,0,OTHER,MONITOR,Lower priority based on baseline
7,content_a63219c6e95a,3,VISIBLE,MONITOR,Lower priority based on baseline
8,content_5e6c160719bc,4,VISIBLE,REVIEW_REFRESH,Meets baseline review threshold
9,content_c27558df2b0c,2,VISIBLE,MONITOR,Lower priority based on baseline


In [21]:
# Rank all pages by baseline score
queue = baseline.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

# Add rank
queue["rank"] = range(1, len(queue) + 1)

print("Ranked queue created.")
print("Total pages:", len(queue))

display(
    queue[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "confidence_note",
            "days_since_last_update",
            "impressions_90d",
            "avg_position",
            "ctr"
        ]
    ].head(20)
)

Ranked queue created.
Total pages: 30000


,rank,content_id,score,reason_code,action,confidence_note,days_since_last_update,impressions_90d,avg_position,ctr
0,1,content_cf56e2e2e282,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,194,61678,19.7,0.15
1,2,content_7368877ea310,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,194,59472,24.8,0.13
2,3,content_1bfaa38ff26c,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,194,25715,22.2,0.23
3,4,content_0a91db491d14,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,193,13299,10.5,0.49
4,5,content_5feee3994adb,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,194,7812,39.0,0.01
5,6,content_c2d929d83eaa,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,193,7558,17.9,0.20
6,7,content_b16bd7307b39,5,STALE_VISIBLE,REVIEW_REFRESH,Meets baseline review threshold,194,4590,31.0,0.00
7,8,content_fe16a55cd13d,5,STALE_VISIBLE,REVIEW_REFRESH,Meets baseline review threshold,194,4556,16.4,0.33
8,9,content_ecb6215e79fd,5,STALE_VISIBLE,REVIEW_REFRESH,Meets baseline review threshold,194,4429,25.3,0.38
9,10,content_928af3e22c80,5,STALE_VISIBLE,REVIEW_REFRESH,Meets baseline review threshold,193,1697,15.8,0.12


In [22]:
import os

# Create output directory if needed
os.makedirs("work/outputs", exist_ok=True)

output_columns = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action",
    "confidence_note",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

queue_output = queue[output_columns].copy()

output_path = "work/outputs/baseline_action_score.csv"

queue_output.to_csv(output_path, index=False)

print("CSV successfully written.")
print("Path:", output_path)
print("Rows:", len(queue_output))

CSV successfully written.
Path: work/outputs/baseline_action_score.csv
Rows: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [23]:
top20 = queue.head(20).copy()

top20["what_would_make_it_wrong"] = np.select(
    [
        top20["reason_code"] == "STALE_VISIBLE",
        top20["reason_code"] == "STALE",
        top20["reason_code"] == "VISIBLE"
    ],
    [
        "The page may already be performing well despite being old and visible.",
        "The page may be stale but have little current search exposure.",
        "The page may have strong visibility but may not actually need content changes."
    ],
    default="The baseline may be prioritizing the page without enough evidence that a refresh is useful."
)

review_columns = [
    "rank",
    "content_id",
    "score",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20[review_columns])

,rank,content_id,score,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,6,REVIEW_REFRESH,STALE_VISIBLE,Multiple strong priority signals,The page may already be performing well despit...
1,2,content_7368877ea310,6,REVIEW_REFRESH,STALE_VISIBLE,Multiple strong priority signals,The page may already be performing well despit...
2,3,content_1bfaa38ff26c,6,REVIEW_REFRESH,STALE_VISIBLE,Multiple strong priority signals,The page may already be performing well despit...
3,4,content_0a91db491d14,6,REVIEW_REFRESH,STALE_VISIBLE,Multiple strong priority signals,The page may already be performing well despit...
4,5,content_5feee3994adb,6,REVIEW_REFRESH,STALE_VISIBLE,Multiple strong priority signals,The page may already be performing well despit...
5,6,content_c2d929d83eaa,6,REVIEW_REFRESH,STALE_VISIBLE,Multiple strong priority signals,The page may already be performing well despit...
6,7,content_b16bd7307b39,5,REVIEW_REFRESH,STALE_VISIBLE,Meets baseline review threshold,The page may already be performing well despit...
7,8,content_fe16a55cd13d,5,REVIEW_REFRESH,STALE_VISIBLE,Meets baseline review threshold,The page may already be performing well despit...
8,9,content_ecb6215e79fd,5,REVIEW_REFRESH,STALE_VISIBLE,Meets baseline review threshold,The page may already be performing well despit...
9,10,content_928af3e22c80,5,REVIEW_REFRESH,STALE_VISIBLE,Meets baseline review threshold,The page may already be performing well despit...


### Top-20 review observations

The top-20 queue is a decision-support list rather than a claim that these pages definitely need refreshing.

The baseline prioritizes pages using staleness, search visibility, and average position. Each recommendation therefore needs human review before an actual content change is made.

The "what would make it wrong" column records an important limitation of the rule: the signals can identify pages worth checking, but they cannot determine by themselves whether a content refresh will improve performance.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [24]:
# Identify potentially weak picks for manual inspection
weak_picks = queue[
    (queue["action"] == "REVIEW_REFRESH") &
    (
        (queue["impressions_90d"] < 500) |
        (queue["avg_position"] <= 10)
    )
].head(10)

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "impressions_90d",
            "avg_position",
            "days_since_last_update",
            "ctr"
        ]
    ]
)

,rank,content_id,score,reason_code,action,impressions_90d,avg_position,days_since_last_update,ctr
2590,2591,content_e3ff1b093148,4,STALE_VISIBLE,REVIEW_REFRESH,1408,7.8,183,0.28
2591,2592,content_7f116ae1f6f5,4,STALE_VISIBLE,REVIEW_REFRESH,954,9.0,301,0.42
2592,2593,content_72496874f806,4,STALE_VISIBLE,REVIEW_REFRESH,821,5.8,301,0.24
2593,2594,content_f6fdf87348f6,4,STALE,REVIEW_REFRESH,2,32.5,373,0.00
2594,2595,content_8d56efff1e71,4,STALE,REVIEW_REFRESH,1,35.0,372,0.00


In [25]:
# Check that the baseline did not use outcome/label-derived fields

features_used = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position"
]

label_like_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("Features used by baseline:")
print(features_used)

print("\nLabel-like fields NOT used:")
for col in label_like_fields:
    print(f"{col}: {'present in dataset but NOT used' if col in df.columns else 'not present'}")

print("\nLeakage check:")
print("PASS - the baseline score was built only from pre-decision observable signals.")

Features used by baseline:
['days_since_last_update', 'impressions_90d', 'avg_position']

Label-like fields NOT used:
trend_direction: present in dataset but NOT used
trend_pct: present in dataset but NOT used
is_declining_label: not present

Leakage check:
PASS - the baseline score was built only from pre-decision observable signals.


### Weak picks and leakage check

Some pages may be weak recommendations even when they receive a high baseline score. For example, a highly visible page may already be performing reasonably well, so visibility alone does not mean that a refresh is necessary.

These pages should be treated as candidates for human review rather than automatic refresh decisions.

The baseline does not use `trend_direction`, `trend_pct`, or other outcome-derived fields in its score. It uses only signals that are observable at the decision moment: `days_since_last_update`, `impressions_90d`, and `avg_position`.

Therefore, the baseline is intended as a simple pre-decision ranking rather than a prediction based on the outcome.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.